<a href="https://colab.research.google.com/github/f-ai0/ds-training/blob/main/week7/Day3_Dataset_DataLoader_CNNs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q torch torchvision

Data prep from Day 2, needed for Tasks 3.1–3.2

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

df = pd.read_excel('Practice_Dataset.xlsx')
features = ['punch_count', 'hours_worked', 'satisfaction_score', 'monthly_salary']

X = SimpleImputer(strategy='median').fit_transform(df[features])
y = df['is_absent'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f'X_train shape: {X_train.shape}')

X_train shape: (288, 4)


# Task 3.1 — Custom Dataset Class

In [3]:
from torch.utils.data import Dataset, DataLoader

# A Dataset needs exactly 3 methods: __init__, __len__, __getitem__
class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = TabularDataset(X_train, y_train)
test_dataset = TabularDataset(X_test, y_test)

# DataLoader handles batching, shuffling, and parallel loading
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Look at one batch
for batch_X, batch_y in train_loader:
    print(f'Batch X shape: {batch_X.shape}, batch y shape: {batch_y.shape}')
    break

print(f'Number of batches per epoch: {len(train_loader)}')

Batch X shape: torch.Size([16, 4]), batch y shape: torch.Size([16, 1])
Number of batches per epoch: 18


# Task 3.2 — Training with Batches

In [4]:
class AbsenceClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.net(x)

model = AbsenceClassifier(input_dim=X_train.shape[1]).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(30):
    model.train()
    epoch_loss = 0.0

    for batch_X, batch_y in train_loader:      # loop over BATCHES now
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}: avg_loss={avg_loss:.4f}')

Epoch 10: avg_loss=0.1848
Epoch 20: avg_loss=0.0611
Epoch 30: avg_loss=0.0302


# Task 3.3 — CNN on Fashion-MNIST

In [5]:
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F

transform = transforms.Compose([
    transforms.ToTensor(),                    # converts to tensor AND scales to 0-1
    transforms.Normalize((0.5,), (0.5,)),     # normalize to -1 to 1
])

train_data = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform)
test_data = torchvision.datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
test_loader = DataLoader(test_data, batch_size=128, shuffle=False)

class FashionCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3)   # 1 input channel (grayscale)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3)
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(64 * 5 * 5, 10)           # 10 classes

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # 28->26->13
        x = self.pool(F.relu(self.conv2(x)))   # 13->11->5
        x = x.flatten(1)                       # flatten all but batch dim
        x = self.dropout(x)
        return self.fc(x)

cnn = FashionCNN().to(device)
criterion = nn.CrossEntropyLoss()   # includes softmax internally
optimizer = torch.optim.Adam(cnn.parameters(), lr=0.001)
print(cnn)

100%|██████████| 26.4M/26.4M [00:02<00:00, 12.7MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 202kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.71MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 11.6MB/s]

FashionCNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout): Dropout(p=0.4, inplace=False)
  (fc): Linear(in_features=1600, out_features=10, bias=True)
)


# Task 3.4 — Train and Evaluate the CNN

In [6]:
for epoch in range(5):
    cnn.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = cnn(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    # Evaluate
    cnn.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            preds = cnn(images).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    print(f'Epoch {epoch+1}: loss={running_loss/len(train_loader):.4f}, ',
          f'test_acc={correct/total:.4f}')

# Compare this final test_acc to your Week 6 TensorFlow CNN's test accuracy
# on the same Fashion-MNIST task.

Epoch 1: loss=0.5645,  test_acc=0.8549
Epoch 2: loss=0.3924,  test_acc=0.8726
Epoch 3: loss=0.3529,  test_acc=0.8811
Epoch 4: loss=0.3246,  test_acc=0.8836
Epoch 5: loss=0.3063,  test_acc=0.8960


In [7]:
# Compare this final test_acc to your Week 6 TensorFlow CNN's test accuracy
# on the same Fashion-MNIST task.
print("\nComparison to Week 6 TensorFlow CNN:")
print("  TensorFlow: Test accuracy = 0.8965 (after 10 epochs)")
print(f"  PyTorch:    Test accuracy = {correct/total:.4f} (after 5 epochs)")
print("  Both models reached nearly identical accuracy, but PyTorch got there")
print("  in half the epochs, suggesting a slightly more efficient architecture")
print("  or optimizer setup for this task.")


Comparison to Week 6 TensorFlow CNN:
  TensorFlow: Test accuracy = 0.8965 (after 10 epochs)
  PyTorch:    Test accuracy = 0.8960 (after 5 epochs)
  Both models reached nearly identical accuracy, but PyTorch got there
  in half the epochs, suggesting a slightly more efficient architecture
  or optimizer setup for this task.


# Comparison to Week 6 TensorFlow CNN:

  - **TensorFlow:** Test accuracy = 0.8965 (after 10 epochs)
  - **PyTorch:**   Test accuracy = 0.8960 (after 5 epochs)
  
  Both models reached nearly identical accuracy, but PyTorch got there
  in half the epochs, suggesting a slightly more efficient architecture
  or optimizer setup for this task.